<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/IDB/W10/week_10_practical_session_Q.1%20.ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from dataclasses import dataclass
from pathlib import Path
import zipfile
import subprocess
import sys

In [2]:
# --------------------------------------------------
# PART 1: DATA STRUCTURES
# --------------------------------------------------

@dataclass
class Transaction:
    sequence: int
    transaction_id: str
    timestamp: int
    operation: str
    data_item: str


@dataclass
class DataItem:
    name: str
    read_timestamp: int = 0
    write_timestamp: int = 0



In [3]:

# --------------------------------------------------
# PART 2: SAMPLE DATASET
# --------------------------------------------------

transactions = [
    Transaction(1, "T1", 10, "READ", "Q"),
    Transaction(2, "T2", 20, "WRITE", "Q"),
    Transaction(3, "T3", 15, "READ", "Q"),
    Transaction(4, "T4", 25, "WRITE", "Q"),
    Transaction(5, "T5", 18, "WRITE", "Q")
]

In [4]:
# --------------------------------------------------
# PART 3: READ OPERATION
# --------------------------------------------------

def read_operation(transaction, item):

    ts = transaction.timestamp

    if ts < item.write_timestamp:

        return "REJECT - ROLLBACK"

    else:

        item.read_timestamp = max(
            item.read_timestamp,
            ts
        )

        return "GRANT"

In [5]:
# --------------------------------------------------
# PART 4: WRITE OPERATION
# --------------------------------------------------

def write_operation(transaction, item):

    ts = transaction.timestamp

    if ts < item.read_timestamp:

        return "REJECT - ROLLBACK"

    elif ts < item.write_timestamp:

        return "REJECT - ROLLBACK"

    else:

        item.write_timestamp = ts

        return "GRANT"

In [6]:
# --------------------------------------------------
# PART 5: PROCESS TRANSACTIONS
# --------------------------------------------------

def process_transaction(transaction, item):

    if transaction.data_item != item.name:
        return "ERROR - UNKNOWN DATA ITEM"

    if transaction.operation == "READ":

        return read_operation(transaction, item)

    elif transaction.operation == "WRITE":

        return write_operation(transaction, item)

    else:

        return "ERROR - UNKNOWN OPERATION"

In [7]:
# --------------------------------------------------
# PART 6: EXECUTION LOG
# --------------------------------------------------

def execute_protocol():

    item = DataItem("Q")

    results = []

    results.append(
        "TIMESTAMP-BASED CONCURRENCY CONTROL"
    )

    results.append(
        "=" * 100
    )

    results.append(
        f"Initial R-TS(Q): {item.read_timestamp}"
    )

    results.append(
        f"Initial W-TS(Q): {item.write_timestamp}"
    )

    results.append("")

    header = (
        f"{'Seq':<6}"
        f"{'Transaction':<15}"
        f"{'Timestamp':<12}"
        f"{'Operation':<12}"
        f"{'Item':<8}"
        f"{'Decision':<24}"
        f"{'R-TS(Q)':<12}"
        f"{'W-TS(Q)':<12}"
    )

    results.append(header)
    results.append("-" * 100)

    for transaction in transactions:

        decision = process_transaction(
            transaction,
            item
        )

        row = (
            f"{transaction.sequence:<6}"
            f"{transaction.transaction_id:<15}"
            f"{transaction.timestamp:<12}"
            f"{transaction.operation:<12}"
            f"{transaction.data_item:<8}"
            f"{decision:<24}"
            f"{item.read_timestamp:<12}"
            f"{item.write_timestamp:<12}"
        )

        results.append(row)

    results.append("-" * 100)

    results.append(
        f"Final R-TS(Q): {item.read_timestamp}"
    )

    results.append(
        f"Final W-TS(Q): {item.write_timestamp}"
    )

    return "\n".join(results)

In [8]:
# --------------------------------------------------
# PART 7: CREATE TERMINAL SCREENSHOT
# --------------------------------------------------

def create_screenshot(output_text, output_dir):

    try:

        from PIL import Image, ImageDraw, ImageFont

    except ImportError:

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "pillow"
        ])

        from PIL import Image, ImageDraw, ImageFont

    font = ImageFont.load_default()

    lines = [
        "Google Colab - Timestamp Protocol",
        "$ python timestamp_protocol.py"
    ] + output_text.splitlines()

    width = 1250
    line_height = 25
    height = len(lines) * line_height + 50

    image = Image.new(
        "RGB",
        (width, height),
        "#101418"
    )

    draw = ImageDraw.Draw(image)

    draw.rectangle(
        (0, 0, width, 35),
        fill="#252a31"
    )

    y = 45

    for line in lines:

        if "GRANT" in line:
            color = "#98c379"

        elif "REJECT" in line:
            color = "#e06c75"

        else:
            color = "#ffffff"

        draw.text(
            (20, y),
            line,
            fill=color,
            font=font
        )

        y += line_height

    image.save(
        output_dir / "timestamp_protocol_output.png"
    )